In [ ]:
!pip install rouge-score

In [13]:
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer import losses
from sentence_transformers.sentence_transformer.datasets import DenoisingAutoEncoderDataset
from transformers import AutoTokenizer

# 加载编码器
embedding_model = SentenceTransformer("tsdae_embedding_model", device="cuda")

# 初始化解码器结构
train_loss = losses.DenoisingAutoEncoderLoss(
    embedding_model,
    decoder_name_or_path="bert-base-uncased",
    tie_encoder_decoder=False
)

# 加载已保存的解码器权重
decoder_path = "tsdae_embedding_model/decoder.pt"
train_loss.decoder.load_state_dict(torch.load(decoder_path, map_location="cuda"))
train_loss.decoder = train_loss.decoder.to("cuda")
train_loss.decoder.eval()

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def reconstruct_sentence(damaged_sentence: str, max_length=64) -> str:
    with torch.no_grad():
        embedding = embedding_model.encode(
            damaged_sentence,
            convert_to_tensor=True,
            device="cuda"
        ).unsqueeze(0).unsqueeze(1)  # [1, 1, hidden_dim]

        input_ids = torch.tensor([[tokenizer.cls_token_id]], device="cuda")

        for _ in range(max_length):
            outputs = train_loss.decoder(
                input_ids=input_ids,
                encoder_hidden_states=embedding,
            )
            next_token_id = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            if next_token_id.item() == tokenizer.sep_token_id:
                break
            input_ids = torch.cat([input_ids, next_token_id], dim=-1)

        return tokenizer.decode(input_ids[0][1:], skip_special_tokens=True)


# 测试
mnli = load_dataset("nyu-mll/glue", "mnli", split="train").select(range(3))
test_sentences = list(mnli["premise"]) + list(mnli["hypothesis"])
damaged_dataset = DenoisingAutoEncoderDataset(test_sentences)

for data in damaged_dataset:
    damaged, original = data.texts[0], data.texts[1]
    reconstructed = reconstruct_sentence(damaged)
    print(f"Damaged:  {damaged}")
    print(f"Original: {original}")
    print(f"Rebuilt:  {reconstructed}")
    print("-" * 60)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertLMHeadModel LOAD REPORT from: bert-base-uncased
Key                                                                | Status     | 
-------------------------------------------------------------------+------------+-
bert.pooler.dense.bias                                             | UNEXPECTED | 
bert.pooler.dense.weight                                           | UNEXPECTED | 
cls.seq_relationship.bias                                          | UNEXPECTED | 
cls.seq_relationship.weight                                        | UNEXPECTED | 
bert.encoder.layer.{0...11}.crossattention.output.dense.bias       | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.output.LayerNorm.weight | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.key.bias           | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.query.bias         | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.value.weight       | MISSING    | 
bert.encoder.layer.{

Damaged:  has two dimensions and geography.
Original: Conceptually cream skimming has two basic dimensions - product and geography.
Rebuilt:  the two dimensions of the two dimensions have a dual dimension.
------------------------------------------------------------
Damaged:  during the and i guess your level uh lose to the next level the parent team decide to a A guy up him a replace
Original: you know during the season and i guess at at your level uh you lose them to the next level if if they decide to recall the the parent team the Braves decide to call to recall a guy from triple A then a double A guy goes up to replace him and a single A guy goes up to replace him
Rebuilt:  and uh i think the first one i get to get to the first one to get a new one and then i get to get a new one to get a new one and then i get to the next one and then i get to get the first one to get a new one and then i get to get the next
------------------------------------------------------------
Damaged:  O